<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **1. Control and Coordination in Mobile Systems**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 1.1 Introduction

Control theory is about modeling how a system evolves, deciding how to influence it, and ensuring it behaves predictably and stably. Control and coordination in mobile and distributed systems can be intuitively introduced through an example that is both familiar and safety-critical: the insulin pump. The pump continuously measures blood glucose levels and adjusts insulin delivery in response. It is a clear case of a feedback control system: one that must sense, decide, and act based on incomplete and delayed information, all while ensuring stability and safety. The insulin pump is computational software embedded in a network of sensors, actuators, and algorithms. It does not simply follow a pre-programmed schedule but reacts to changing conditions in the body, much like mobile devices and distributed systems respond to changes in their environments. Each measurement can be noisy, each control action delayed, and yet the system must remain stable and converge to the desired glucose level. Understanding such a system means understanding feedback: how a device can regulate a physical process using data that reflects its own previous actions. When we generalize this to a collection of devices (each with sensors, processors, and communication links) we arrive at the idea of distributed control. Here, decisions emerge from cooperation and information exchange rather than a single central authority. This perspective leads naturally to the study of stability, convergence, and optimality in networked systems. It also highlights the practical importance of communication, since every feedback signal must pass through a channel that may be noisy, lossy, or delayed. Whether we consider an insulin pump communicating with a continuous glucose monitor, a swarm of drones coordinating through wireless links, or a mobile app adapting its sensing rate to conserve energy, the same principles apply.

In this notebook, by examining the insulin pump as a guiding example, we learn how feedback can be used to produce reliable behavior in uncertain environments. This understanding forms the basis for designing and analyzing control strategies in all types of mobile and distributed systems.

### 1.2 Linear control systems with and without feedback

Feedback occurs when two or more dynamic systems interact so that each influences the other. When this interaction forms a closed cycle, the system is called closed-loop; when it is broken, the system operates open-loop. Control is the systematic use of feedback and algorithms to influence system behavior. Every feedback loop involves three essential processes: sensing, computation, and actuation. Sensing means measuring the system’s state, such as blood glucose. Computation means comparing the measured value with the desired target and deciding on corrective action. Actuation means applying an input to influence the system, for example by injecting insulin. The goal of control design is to make the closed-loop system stable, meaning that bounded disturbances should produce bounded errors, and to achieve the desired performance, such as quick response to changes and strong rejection of disturbances. Two important advantages of feedback are robustness to uncertainty and the ability to shape the dynamics of a system. Feedback can maintain the desired operation even when system parameters vary, and it can make a system faster, smoother, or more stable. Poorly designed feedback can cause instability, amplify noise, or create unwanted couplings between subsystems.

The artificial pancreas is a modern example of feedback control, where sensing, computation, and actuation work together in a loop that maintains stability and safety in the presence of continuous biological and environmental disturbances. The figure below illustrates how feedback control applies to blood glucose regulation. At the top, the closed-loop artificial pancreas automatically adjusts insulin delivery. A glucose sensor, or continuous glucose monitor (CGM), continuously measures blood glucose levels and sends this data to a controller algorithm. The controller compares the measured glucose with the target setpoint and computes the insulin dose required to minimize the error. This insulin is delivered through a pump and cannula into the body, which changes the blood glucose concentration through the body’s natural metabolism. The updated glucose level is then measured again, closing the feedback loop. External disturbances such as meals, exercise, and stress constantly affect the system, but feedback control compensates for these effects automatically. In contrast, the open-loop system, shown at the bottom of the figure, relies on manual user input. The user measures glucose, estimates the required insulin based on experience or tables, and delivers insulin manually through the pump. This system has no automatic correction; errors can persist longer, and control depends entirely on the user’s actions.


<center>
<img src="../../.admin/notebooks/picture/insulin01.png" 
     alt="Artificial Pancreas" 
     width="90%" 
     style="filter: grayscale(60%);">
     <p><em>Closed loop and open loop control of insulin delivery. The more advanced controller algorithms such as a proportional–integral–derivative (PID) or model predictive controller (MPC) are explained in another notebook.</em></p>
</center>



#### 1.2.1 Closed loop control with feedback loop


Let $x_t$ be the **state**, representing the body’s internal glucose level at time $t$, and let $u_t$ be the **control input**, the amount of insulin delivered by the pump. Then the system model is simply:


$$
x_{t+1} = a x_t + b u_t
$$


In this simple linear approach, $a$ determines how much of the current glucose level carries over to the next time step, and $b$ determines how strongly the delivered insulin affects glucose reduction. For instance, if $a = 0.9$ and $b = 0.1$, the glucose level decreases slowly, meaning the body’s natural glucose regulation is relatively inert and insulin acts gradually.


Now suppose the control law is based on the error between a desired glucose value $x^*$ (the target) and the actual measured value $x_t$:


$$
u_t = k(x^* - x_t)
$$


Substituting this control law into the system model gives


$$
x_{t+1} = a x_t + b k (x^* - x_t)
$$


which simplifies to


$$
x_{t+1} = (a - b k) x_t + b k x^*
$$


Here, the **feedback gain $k$** represents how aggressively the artificial pancreas delivers insulin in response to deviations from the target glucose level. The parameter $a$ describes the body’s glucose persistence (its inertia), $b$ quantifies insulin sensitivity, and $k$ adjusts the controller’s responsiveness to errors.

#### 1.2.2 Measurement

A more realistic model contains the effect of the measurement and uses the measured glucose in the feedback. It adds an output/measurement $y_t$ (what the CGM reads) with a sensor gain $c$:
$$
\boxed{y_t = c\,x_t}
$$

> Sensor noise can also be included in the model as
> $y_t = c\,x_t + \varepsilon_t$,
> where $\varepsilon_t$ represents measurement noise and quantization effects. This noise is typically modeled as independent additive white Gaussian noise (AWGN).


The controller uses the measured value:
$$
u_t = k\bigl(x^* - y_t\bigr)=k\bigl(x^* - c\,x_t\bigr).
$$

Substitute into $x_{t+1}=a x_t + b u_t$:
$$
x_{t+1}=a x_t + b\,k\bigl(x^* - c\,x_t\bigr)
$$

So the closed-loop scalar dynamics with measurement are
$$
\boxed{x_{t+1}=(a-bkc)\,x_t + b k\,x^*, \qquad y_t=c\,x_t.}
$$

>In modern models, glucose–insulin regulation involves multiple interacting variables. The state $x_t$ may include blood glucose, insulin in plasma, and insulin in tissues; the control input $u_t$ is the insulin infusion rate; and the measured output $y_t$ is the glucose concentration provided by the continuous glucose monitor (CGM). The system model then becomes
>
>$$
>x_{t+1} = A x_t + B u_t \quad \text{and} \quad y_t = C x_t
>$$
>
>Here, $A$ captures how the internal metabolic states evolve over time, $B$ describes how insulin delivery influences them, and $C$ specifies which internal quantities are actually
>measured - for example, the glucose sensor measures only blood glucose, not tissue insulin absorption or hepatic glucose production.

#### 1.2.3 Artificial Pancreas: Meals, sport, and hypoglycemia rescue

The following example illustrates the effect of closed-loop control in an artificial pancreas system. The simulation models glucose dynamics during typical daily events such as meals (which raise glucose) and physical activity (which lowers glucose). The closed-loop feedback, implemented through a controller with a glucagon-like rescue term, demonstrates its clear advantage over open-loop (basal-only) insulin delivery by automatically compensating for these disturbances and preventing hypoglycemia.

>"Hypoglycemia" means low blood glucose (sugar) levels, measured in parts per volume. It occurs when the body’s available glucose falls below what is needed for normal brain and organ function. It might happen if too much insulin is delivered, a meal is missed, or intense physical activity consumes excess glucose. The controller must prevent hypoglycemia by reducing insulin delivery when glucose levels drop too low.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# ---------- parameters ----------
Tsim = 360.0          # total time [min]
dt   = 0.5
t    = np.arange(0, Tsim, dt)

Tg = 50.0             # natural glucose decay [min]
k_ins = 0.8           # how strongly insulin lowers glucose (positive)
r = 0.0               # target deviation

# "glucagon-like" rescue term
k_gluc = 0.4          # bigger -> stronger recovery from hypoglycemia

# events
t_meal1   = 30.0
meal1_amp = 1.2

t_sport_start = 120.0
t_sport_end   = 160.0
sport_amp     = -1.8   # sport consumes glucose

t_meal2   = 180.0
meal2_amp = 0.4

# ---------- noise model ----------
np.random.seed(1)
noise_std = 0.8     # standard deviation of measurement noise

# ---------- storage ----------
x_ol = np.zeros_like(t)    # open loop glucose (true)
x_cl = np.zeros_like(t)    # closed loop glucose (true)
u_cl = np.zeros_like(t)    # control signal (insulin)

# ---------- controller ----------
u_basal = 0.5
Kp, Ki = 0.4, 0.01
I = 0.0

# ---------- simulate ----------
for k in range(1, len(t)):
    tk = t[k]

    # time-varying disturbance from meals and sport
    d = 0.0
    if tk >= t_meal1:
        d += meal1_amp
    if t_sport_start <= tk <= t_sport_end:
        d += sport_amp
    if tk >= t_meal2:
        d += meal2_amp

    # --- open loop (no feedback, no glucagon) ---
    xdot_ol = -(1.0/Tg)*x_ol[k-1] + d - k_ins*u_basal
    x_ol[k] = x_ol[k-1] + dt*xdot_ol

    # --- closed loop (PI + glucagon-like) ---
    y_meas = x_cl[k-1] + np.random.normal(0, noise_std)

    e = y_meas - r
    I += e*dt
    u = u_basal + Kp*e + Ki*I
    u = max(0.0, u)
    u_cl[k] = u

    gluc = k_gluc * max(0.0, -x_cl[k-1])

    xdot_cl = -(1.0/Tg)*x_cl[k-1] + d - k_ins*u + gluc
    x_cl[k] = x_cl[k-1] + dt*xdot_cl

# ---------- plot ----------
plt.figure(figsize=(12, 4.5))
plt.plot(t, x_ol, label="Open-loop (basal only)")
plt.plot(t, x_cl, label="Closed-loop (PI + glucagon-like)")

# Mark meal and sport events
plt.axvline(t_meal1, linestyle=":", color="tab:blue", label="Meal 1")
plt.axvline(t_meal2, linestyle=":", color="tab:orange", label="Meal 2")

# Shade the sport interval
plt.axvspan(t_sport_start, t_sport_end, color="tab:purple", alpha=0.2, label="Sport phase")

# Target line
plt.axhline(0.0, linestyle="--", color="gray", label="Target")

plt.xlabel("Time [min]")
plt.ylabel("Glucose deviation [arb. units]")
plt.title("Artificial pancreas with noisy glucose measurement")
plt.legend(loc="best")
plt.tight_layout()
plt.show()


In the graph above, the open-loop curve shows how the glucose level changes when the insulin pump delivers only a constant background amount of insulin ("basal"), without any automatic feedback or corrective action. In this mode, glucose simply follows the body’s natural processes together with any external influences such as meals or physical activity.

At the beginning of the simulation, no food has been consumed, so the constant insulin supply acts alone. This slowly lowers the glucose level even though it starts at the desired value, leading to a gradual drop similar to mild hypoglycemia caused by too much insulin without food intake. When the first meal occurs, glucose increases for a while because of the new energy intake. During the sport phase, glucose is consumed more quickly, so the level falls again. After the second meal, the intake of glucose dominates once more, and since the system has no feedback to adjust insulin delivery, the glucose level rises steadily toward the end of the simulation.

The open-loop result therefore reflects how glucose is influenced by insulin, meals, and physical activity in the absence of regulation. In contrast, the closed-loop system automatically adapts insulin delivery based on the measured glucose level, reducing insulin when glucose is low and increasing it when glucose is high. This keeps the glucose level much closer to the desired range and prevents both low and high extremes.

>In our context, **basal** means *background* or *steady*. The basal insulin rate is the small, continuous amount of insulin that an insulin pump delivers throughout the day and night to cover the body’s normal metabolic needs when no food is being eaten. It keeps glucose levels stable between meals and during sleep by balancing the liver’s slow release of glucose into the bloodstream. In contrast, **bolus insulin** refers to larger, temporary doses given at specific times, such as before meals, to handle the extra glucose coming from food.

#### 1.2.4 System stability

Stability is a fundamental concept in control theory because it determines whether a system behaves predictably or spirals out of control. A stable system reacts to disturbances, noise, or small errors by eventually settling back to a desired operating point. An unstable system, on the other hand, amplifies these deviations over time: Its output grows without bound, oscillates uncontrollably, or collapses entirely. In practical terms, stability means that a feedback loop works as intended by correcting errors instead of making them worse. A system is stable if it does not diverge when disturbed, and instead it returns to equilibrium.

From the above,  $x_{t+1} = (a - b k) x_t + \text{constant}$ is stable if $|a - b k| < 1$. This inequality captures the mathematical idea of control. If feedback is too weak (small $k$), the system is slow to respond. If feedback is too strong (large $k$), the system overshoots or oscillates. And finally, if $|a - b k| > 1$, the system diverges and is unstable.

The code below illustrates the same system with three different feedback strengths. In blue, feedback is too weak, and the system takes forever to settle. In green, feedback is well tuned and the response decays quickly. In red, feedback is too aggressive, so each correction overshoots, and the oscillation grows instead of shrinking. This visualizes the inequality $|a - b k| < 1$: Only when the pole stays inside the unit circle do we get stability.

* For small k: $a - b k ≈ 1$ → very slow decay, stable but sluggish.
* For moderate k: $a - b k ≈ 0.25$ → fast, well-behaved convergence.
* For large k: $a - b k < -1$ → sign alternates and magnitude grows → unstable oscillation.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# Discrete-time model: x_{t+1} = a*x_t + b*u_t, with u_t = -k*(x_t - x_set)
a = 1.0
b = 0.5
x0 = 0.0

# Time steps
steps = 30
t = np.arange(steps)

# Desired (setpoint) value
x_set_final = 1.0     # target value after jump
setpoint_time = 10     # time step when target changes

# Fixed feedback gains
k_values = [0.08, 1.5, 3.5]
labels = [
    "Too weak feedback (slow response)",
    "Good feedback (stable, fast)",
    "Too strong feedback (unstable)"
]
colors = ["tab:blue", "tab:green", "tab:red"]

def simulate_and_plot(k_slider):
    plt.figure(figsize=(12,4.5))

    # Fixed curves
    for k, label, color in zip(k_values, labels, colors):
        x = np.zeros(steps)
        x[0] = x0
        for i in range(1, steps):
            x_set = x_set_final if i >= setpoint_time else 0.0
            u = -k * (x[i-1] - x_set)
            x[i] = a * x[i-1] + b * u
        plt.plot(t, x, marker="o", color=color, label=label)

    # Slider-adjusted curve
    x = np.zeros(steps)
    x[0] = x0
    for i in range(1, steps):
        x_set = x_set_final if i >= setpoint_time else 0.0
        u = -k_slider * (x[i-1] - x_set)
        x[i] = a * x[i-1] + b * u
    plt.plot(t, x, marker="o", color="tab:purple", label=f"Interactive gain k = {k_slider:.2f}")

    # Desired trajectory
    x_set_series = [x_set_final if i >= setpoint_time else 0.0 for i in range(steps)]
    plt.plot(t, x_set_series, 'k--', linewidth=1.5, label="Desired value (setpoint)")

    plt.axhline(0, color="gray", linestyle="--", linewidth=1)
    plt.xlabel("time step t")
    plt.ylabel("state xₜ")
    plt.title("Effect of feedback gain on response to delayed setpoint change")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Create the slider widget
interact(simulate_and_plot, k_slider=widgets.FloatSlider(
    value=1.0, min=0.0, max=5.0, step=0.02,
    description='k (slider)'
));


### 1.3 Remarks [03]

Control and signal theory can be applied to banking, insurance, and national finance by viewing financial systems as dynamical systems that evolve over time and can be influenced through feedback. Concepts such as feedback, estimation, and optimal control provide a mathematical and practical framework for managing uncertainty, reacting to changing conditions, and ensuring system stability.

In banking, control principles can be used for liquidity management, credit risk monitoring, and algorithmic trading. Liquidity management can be formulated as a predictive control problem, where borrowing, lending, or asset sales are adjusted to maintain target reserves while minimizing cost. Credit risk monitoring can use estimation methods to infer the true, time-varying creditworthiness of borrowers. Trading and market-making can be framed as stochastic control problems where inventory and pricing are continuously adjusted to balance profit and risk.

Signal processing and filtering complement these methods by denoising financial time series, detecting anomalies, and identifying regime changes. They serve as the sensing and estimation layers of financial control loops.

In practice, applying these methods involves building a model of the system’s dynamics, estimating hidden states with filters, and designing feedback rules. The system’s performance is then evaluated in simulation and live operation using both financial and control metrics, such as stability, responsiveness, cost efficiency, and risk exposure. Through this perspective, financial decision-making becomes a structured feedback process balancing stability, responsiveness, and robustness in uncertain and dynamic environments.



### 1.4 References
[01] Åström K. & Murray R. [**Feedback Systems**](https://fbswiki.org/wiki/index.php/Feedback_Systems:_An_Introduction_for_Scientists_and_Engineers) Princeton University Press. July 2020.

[02] Hespanha, João P. [**Linear Systems Theory**](https://eth.swisscovery.slsp.ch/permalink/41SLSP_ETH/lshl64/alma99117944466405503) 2nd edition. Princeton, NJ: Princeton University Press, 2018.

[03] Sarychev, A. et al. [**Mathematical Control Theory and Finance**](https://eth.swisscovery.slsp.ch/permalink/41SLSP_ETH/lshl64/alma99117345757905503). Berlin, Heidelberg: Springer, 2008.

----
### 1.5 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**What is the difference between an open-loop and a closed-loop control system?** Answer:  

(add your answer here)

**Try the interactive slider in the graph above. What would be the best gain to be used in this example?** Answer:  

(add your answer here)

**Give one example of an application of control systems in mobile devices or everyday life. In this example, what represents the state and the control input?** Answer:  

(add your answer here)

**Which type of control system is most commonly used in practice, and why?** Answer:  

(add your answer here)

**How can a closed-loop system recover from disturbances or changes in the environment?** Answer:  

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)